# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzaabutt/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains and compares machine learning models against our deterministic baseline to predict content decay (`is_declining_label`) using a client-aware validation split.

## 1. Method choice and why

We evaluate three distinct modeling techniques:
1. **Logistic Regression (Baseline Linear Model)**: Serves as a interpretable linear baseline to test linear separability.
2. **Decision Tree (Max Depth 5)**: Captures basic non-linear interaction rules.
3. **Random Forest Classifier (Ensemble, Selected Model)**: Selected because it handles non-linear SERP position decay, captures multi-way feature interactions (e.g. high impressions + position decay + low CTR), and prevents overfitting via ensemble averaging and sub-sampling.

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
import numpy as np

# Set path relative to repo root
repo_root = Path("../../").resolve() if Path("../../scripts").exists() else Path("../").resolve()
sys.path.append(str(repo_root / "scripts"))
from ml_utils import PROCESSED_DIR, OUTPUT_DIR

# Load model results receipt
results_json_path = OUTPUT_DIR / "model_results.json"
if not results_json_path.exists():
    results_json_path = repo_root / "outputs" / "model_results.json"

with open(results_json_path, "r") as f:
    results = json.load(f)

print(f"Loaded model evaluation results on {results['input_rows']:,} rows.")
print(f"Split Strategy: {results['split_strategy']}")
print(f"Best Model Selected: {results['best_model']['name']}")

## 2. Split design

**Client-Aware Holdout Split (`client_holdout`):**
- Rather than random row-level splitting (which causes data leakage when multiple pages from the same client domain appear in both train and test sets), we split by `client_id`.
- 80% of client domains (27,675 rows) are used for training, while 20% of independent client domains (2,325 rows) are held out exclusively for testing.
- This guarantees that test performance accurately reflects model generalizability to newly onboarded sites.

In [ ]:
print(f"Train Rows: {results['train_rows']:,}")
print(f"Test Rows: {results['test_rows']:,}")
print(f"Target Positive Rate (Declining Pages): {results['target_positive_rate']:.2%}")

## 3. Train + compare vs my baseline

We compare all models against the deterministic Week 4 baseline on the exact same client-holdout split using Precision@50, PR-AUC, and ROC-AUC.

In [ ]:
models_dict = results["models"]
baseline_dict = results["baseline"]

comparison_data = []
comparison_data.append({
    "Model": "Deterministic Baseline (Week 4)",
    "Precision@50": baseline_dict.get("baseline_precision_at_50", 0),
    "Precision@20": baseline_dict.get("baseline_precision_at_20", 0),
    "PR-AUC": baseline_dict.get("baseline_average_precision", 0),
    "ROC-AUC": baseline_dict.get("baseline_roc_auc", 0),
    "Accuracy": baseline_dict.get("baseline_accuracy", 0)
})

for name, m in models_dict.items():
    comparison_data.append({
        "Model": name.replace("_", " ").title(),
        "Precision@50": m.get("precision_at_50", 0),
        "Precision@20": m.get("precision_at_20", 0),
        "PR-AUC": m.get("average_precision", 0),
        "ROC-AUC": m.get("roc_auc", 0),
        "Accuracy": m.get("accuracy", 0)
    })

df_comp = pd.DataFrame(comparison_data)
print("Model vs Baseline Comparison (Client Holdout Split):")
df_comp

## 4. Errors and interpretation

**Feature Importance & Error Analysis:**
- The top predictive feature is `days_with_impressions` (15.8% importance), followed by `log_impressions_90d` (12.9%), `avg_position` (10.9%), and `content_age_days` (9.5%).
- **False Positives**: Occur when pages experience temporary seasonal declines or keyword volatility despite active editorial freshness.
- **False Negatives**: Occur on low-volume niche pages where traffic decay happens subtly over long windows.

In [ ]:
fi_list = results["best_model"]["feature_importance_top"]
df_fi = pd.DataFrame(fi_list).head(10)
print("Top 10 Feature Importances (Random Forest):")
df_fi

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`